# Baseline Gemini Muay Thai Individual Strike Classifier

This notebook evaluates Gemini 2.5 Flash directly (without DSPy) to serve as a baseline comparison for the DSPy classifier on individual strikes. It uses the same data splits and evaluation metrics. Each video contains a single strike, and the strike type is stored in a label.txt file in each directory.


## Section 1: Setup and Configuration


In [1]:
import google.generativeai as genai
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
import time
from typing import List, Dict, Tuple
import re

# Load environment variables
load_dotenv()

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables. Please set it in .env file.")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Initialize Gemini model
try:
    model = genai.GenerativeModel("gemini-2.5-flash")
    print('Using gemini-2.5-flash')
except:
    raise ValueError('failed to load gemini-2.5-flash')
    

# Data paths
DATA_ROOT = Path("data/individual_strikes")

# Training split size (5 videos per directory)
TRAIN_SIZE = 5

# Rate limiting for free tier (10 requests per minute)
# Use 7 seconds between requests to stay safely under limit
REQUEST_DELAY = 7

print("Setup complete!")
print(f"Data directory: {DATA_ROOT}")
print(f"Training videos per strike type: {TRAIN_SIZE}")
print(f"Rate limiting: {REQUEST_DELAY}s delay between API calls")


Using gemini-2.5-flash
Setup complete!
Data directory: data\individual_strikes
Training videos per strike type: 5
Rate limiting: 7s delay between API calls


## Section 2: Data Loading


In [ ]:
def load_individual_strike_examples(data_root: Path) -> List[Dict]:
    """
    Load all individual strike videos from subdirectories.
    Each directory contains a label.txt file with the strike type.
    Returns a list of examples with video_path and strike_type.
    """
    examples = []
    
    # Iterate through all subdirectories in data/individual_strikes
    for strike_dir in sorted(data_root.iterdir()):
        if not strike_dir.is_dir():
            continue
        
        # Read strike type from label.txt file
        label_file = strike_dir / "label.txt"
        if not label_file.exists():
            print(f"Warning: label.txt not found in {strike_dir.name}, skipping...")
            continue
        
        try:
            with open(label_file, 'r') as f:
                strike_type = f.read().strip()
            
            if not strike_type:
                print(f"Warning: label.txt is empty in {strike_dir.name}, skipping...")
                continue
        except Exception as e:
            print(f"Warning: Error reading label.txt in {strike_dir.name}: {e}, skipping...")
            continue
        
        # Get all .mov files in this directory
        video_files = sorted(strike_dir.glob("*.mov"))
        
        for video_path in video_files:
            example = {
                'video_path': str(video_path),
                'strike_type': strike_type  # Label comes from label.txt file
            }
            examples.append(example)
    
    return examples

# Load all examples
all_examples = load_individual_strike_examples(DATA_ROOT)

print(f"Loaded {len(all_examples)} total examples")

# Group examples by strike type
examples_by_strike = {}
for example in all_examples:
    strike_type = example['strike_type']
    if strike_type not in examples_by_strike:
        examples_by_strike[strike_type] = []
    examples_by_strike[strike_type].append(example)

print(f"\nExamples per strike type:")
for strike_type in sorted(examples_by_strike.keys()):
    count = len(examples_by_strike[strike_type])
    print(f"  {strike_type}: {count} videos")


Loaded 336 total examples

Examples per strike type:
  cross: 24 videos
  jab: 24 videos
  knee: 24 videos
  lead_elbow: 24 videos
  lead_hook: 24 videos
  lead_uppercut: 24 videos
  low_kick: 24 videos
  rear_elbow: 24 videos
  rear_hook: 24 videos
  rear_uppercut: 24 videos
  roundhouse: 24 videos
  switch_kick: 24 videos
  switch_knee: 24 videos
  teep: 24 videos


In [3]:
# Split data into train and validation
# Use the same random seed (42) as DSPy notebook for reproducibility
# Select 5 videos per strike type for training, rest for validation

random.seed(42)  # Same seed as other notebooks

train_examples = []
val_examples = []

# Split each strike type separately
for strike_type in sorted(examples_by_strike.keys()):
    strike_examples = examples_by_strike[strike_type]
    
    # Shuffle the list of videos for this strike type
    indices = list(range(len(strike_examples)))
    random.shuffle(indices)
    
    # Select first TRAIN_SIZE for training
    train_indices = set(indices[:TRAIN_SIZE])
    train_set = [strike_examples[i] for i in train_indices]
    val_set = [strike_examples[i] for i in range(len(strike_examples)) if i not in train_indices]
    
    train_examples.extend(train_set)
    val_examples.extend(val_set)
    
    print(f"{strike_type}: {len(train_set)} train, {len(val_set)} val")

print(f"\nTraining set: {len(train_examples)} examples")
print(f"Validation set: {len(val_examples)} examples")

# Display statistics
unique_strikes = set(ex['strike_type'] for ex in train_examples + val_examples)
print(f"\nUnique strike types: {sorted(unique_strikes)}")
print(f"Total unique strike types: {len(unique_strikes)}")


cross: 5 train, 19 val
jab: 5 train, 19 val
knee: 5 train, 19 val
lead_elbow: 5 train, 19 val
lead_hook: 5 train, 19 val
lead_uppercut: 5 train, 19 val
low_kick: 5 train, 19 val
rear_elbow: 5 train, 19 val
rear_hook: 5 train, 19 val
rear_uppercut: 5 train, 19 val
roundhouse: 5 train, 19 val
switch_kick: 5 train, 19 val
switch_knee: 5 train, 19 val
teep: 5 train, 19 val

Training set: 70 examples
Validation set: 266 examples

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_uppercut', 'low_kick', 'rear_elbow', 'rear_hook', 'rear_uppercut', 'roundhouse', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 14


## Section 3: Direct Gemini Classification


In [7]:
def normalize_strike_name(strike: str) -> str:
    """
    Normalize strike name to match directory naming convention.
    Handles variations like spaces, hyphens, case differences.
    """
    # Convert to lowercase and replace spaces/hyphens with underscores
    normalized = strike.lower().strip()
    normalized = normalized.replace(' ', '_').replace('-', '_')
    return normalized

def classify_video_baseline(video_path: str, model=model) -> str:
    """
    Classify the single Muay Thai strike in a video using Gemini API directly.
    Returns a single strike name string (not array).
    """
    prompt = f"""Watch this Muay Thai training video and identify the single strike being performed. 
This video contains only one strike. Return your answer as just the strike name, potential strikes: {unique_strikes}.
Only return the strike name, nothing else."""
    
    try:
        # Load video file
        video_file = genai.upload_file(path=video_path)
        
        # Retrieve the file to check its state
        myfile = genai.get_file(name=video_file.name)
        
        # Wait for the file to be in an ACTIVE state
        while myfile.state.name != 'ACTIVE':
            print(f'File state is {myfile.state.name}. Waiting...')
            time.sleep(5)  # Wait for 5 seconds before checking again
            myfile = genai.get_file(name=video_file.name)  # Get the updated file status
        
        print(f'File {myfile.name} is now {myfile.state.name}.')
        
        # Generate content
        response = model.generate_content([prompt, video_file])
        
        # Parse response
        response_text = response.text.strip()
        
        # Clean up the response - remove quotes, brackets, etc.
        response_text = response_text.strip('"\'[]')
        
        # Normalize the strike name
        predicted_strike = normalize_strike_name(response_text)
        
        return predicted_strike
        
    except Exception as e:
        print(f"Error classifying video {video_path}: {e}")
        return ""

print("Baseline classification function created!")


Baseline classification function created!


## Section 4: Evaluation


In [8]:
# Evaluation metrics for individual strikes

def exact_match_metric(true_strike: str, predicted_strike: str) -> bool:
    """Evaluate exact match between predicted and true strike."""
    # Normalize both for comparison
    true_normalized = normalize_strike_name(true_strike)
    pred_normalized = normalize_strike_name(predicted_strike)
    return true_normalized == pred_normalized

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [9]:
# Evaluate on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")
print(f"Using {REQUEST_DELAY}s delay between requests to avoid rate limits\n")

results = []
exact_matches = 0

for i, example in enumerate(val_examples):
    try:
        # Add rate limiting delay (except for first request)
        if i > 0:
            print(f"Waiting {REQUEST_DELAY}s to avoid rate limits...")
            time.sleep(REQUEST_DELAY)
        
        print(f"Processing {i+1}/{len(val_examples)}: {Path(example['video_path']).name} (true: {example['strike_type']})")
        
        # Classify video
        predicted_strike = classify_video_baseline(example['video_path'])
        
        true_strike = example['strike_type']
        exact_match = exact_match_metric(true_strike, predicted_strike)
        
        if exact_match:
            exact_matches += 1
        
        results.append({
            'video_path': example['video_path'],
            'true_strike': true_strike,
            'predicted_strike': predicted_strike,
            'exact_match': exact_match
        })
        
        print(f"  True: {true_strike}")
        print(f"  Pred: {predicted_strike}")
        print(f"  Match: {'✓' if exact_match else '✗'}\n")
            
    except Exception as e:
        print(f"Error processing {example['video_path']}: {e}\n")
        results.append({
            'video_path': example['video_path'],
            'true_strike': example['strike_type'],
            'predicted_strike': "",
            'exact_match': False,
            'error': str(e)
        })

print("Evaluation complete!")


Evaluating on 266 validation examples...
Using 7s delay between requests to avoid rate limits

Processing 1/266: cross_side_bag_garage00000822_V1-0004.mov (true: cross)
File state is PROCESSING. Waiting...
File files/2dm2swetqn3o is now ACTIVE.
  True: cross
  Pred: lead_hook
  Match: ✗

Waiting 7s to avoid rate limits...
Processing 2/266: cross_side_bag_garage00000854_V1-0005.mov (true: cross)
File state is PROCESSING. Waiting...
File files/83dxgymwt7y9 is now ACTIVE.
  True: cross
  Pred: rear_hook
  Match: ✗

Waiting 7s to avoid rate limits...
Processing 3/266: cross_side_bag_garage00000884_V1-0006.mov (true: cross)
File state is PROCESSING. Waiting...
File files/hq6xrtps92de is now ACTIVE.
  True: cross
  Pred: cross
  Match: ✓

Waiting 7s to avoid rate limits...
Processing 4/266: cross_side_bag_garage00005887_V1-0126.mov (true: cross)
File state is PROCESSING. Waiting...
File files/1bb2mgmf8amd is now ACTIVE.
  True: cross
  Pred: jab
  Match: ✗

Waiting 7s to avoid rate limits...

In [10]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:10], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_strike']}")
    print(f"  Pred:  {result['predicted_strike']}")
    print(f"  Match: {'✓' if result['exact_match'] else '✗'}")


EVALUATION RESULTS
Exact Match Accuracy: 13.16% (35/266)

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: cross_side_bag_garage00000822_V1-0004.mov
  True:  cross
  Pred:  lead_hook
  Match: ✗

Example 2:
  Video: cross_side_bag_garage00000854_V1-0005.mov
  True:  cross
  Pred:  rear_hook
  Match: ✗

Example 3:
  Video: cross_side_bag_garage00000884_V1-0006.mov
  True:  cross
  Pred:  cross
  Match: ✓

Example 4:
  Video: cross_side_bag_garage00005887_V1-0126.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 5:
  Video: cross_side_bag_garage00005921_V1-0127.mov
  True:  cross
  Pred:  roundhouse
  Match: ✗

Example 6:
  Video: cross_side_bag_gym00001498_V1-0001.mov
  True:  cross
  Pred:  cross
  Match: ✓

Example 7:
  Video: cross_side_bag_gym00001579_V1-0002.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 8:
  Video: cross_side_bag_gym00001645_V1-0003.mov
  True:  cross
  Pred:  roundhouse
  Match: ✗

Example 9:
  Video: cro

In [11]:
# Calculate per-strike-type accuracy
from collections import defaultdict

strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_strike = result['true_strike']
    pred_strike = result['predicted_strike']
    
    # Count true strikes
    strike_stats[true_strike]['total'] += 1
    
    # Count predicted strikes (if not empty)
    if pred_strike:
        strike_stats[pred_strike]['predicted'] += 1
    
    # Count correct predictions
    if result['exact_match']:
        strike_stats[true_strike]['correct'] += 1

print("\nPer-Strike-Type Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike-Type Accuracy:
------------------------------------------------------------
  cross               : 15.79% (3/19 correct, 6 predicted)
  jab                 : 31.58% (6/19 correct, 11 predicted)
  knee                : 78.95% (15/19 correct, 82 predicted)
  lead_elbow          :  0.00% (0/19 correct, 0 predicted)
  lead_hook           :  0.00% (0/19 correct, 1 predicted)
  lead_uppercut       :  0.00% (0/19 correct, 0 predicted)
  low_kick            :  0.00% (0/19 correct, 1 predicted)
  rear_elbow          :  0.00% (0/19 correct, 0 predicted)
  rear_hook           :  0.00% (0/19 correct, 4 predicted)
  rear_uppercut       :  0.00% (0/19 correct, 0 predicted)
  roundhouse          :  5.26% (1/19 correct, 48 predicted)
  switch_kick         :  0.00% (0/19 correct, 1 predicted)
  switch_knee         :  5.26% (1/19 correct, 8 predicted)
  teep                : 47.37% (9/19 correct, 104 predicted)


## Section 5: Results and Comparison


In [12]:
# Save evaluation results (same format as other baseline notebooks)
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "individual_baseline_evaluation_results.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'individual_baseline_evaluation_results.json'}")


Results saved to results\individual_baseline_evaluation_results.json


In [13]:
# Display summary
print("\n" + "=" * 60)
print("BASELINE EVALUATION SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash (Direct API)")
print(f"Approach: Baseline (no DSPy, no few-shot examples)")
print(f"Dataset: Individual Strikes (one strike per video)")
print(f"Training examples: {len(train_examples)} ({TRAIN_SIZE} per strike type)")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"\nResults saved to: {output_dir}/individual_baseline_evaluation_results.json")
print("\nNote: Compare these results with DSPy classifier results for evaluation.")
print("=" * 60)



BASELINE EVALUATION SUMMARY
Model: Gemini 2.5 Flash (Direct API)
Approach: Baseline (no DSPy, no few-shot examples)
Dataset: Individual Strikes (one strike per video)
Training examples: 70 (5 per strike type)
Validation examples: 266

Final Performance:
  Exact Match Accuracy: 13.16%

Results saved to: results/individual_baseline_evaluation_results.json

Note: Compare these results with DSPy classifier results for evaluation.
